# AROGYA Paper N1 – Google Colab notebook
**Spike encodings and a two-timescale SNN for RUL prediction on NASA C-MAPSS**

Run the cells **from top to bottom**. Each figure and table appears right under the cell that creates it, together with the place where it goes in the manuscript.

**Before you start**
1. Put your GitHub link into `REPO_URL` in Cell A (or upload `AROGYA_N1_code.zip` to the top level of *My Drive*).
2. In Colab choose **Runtime → Change runtime type → CPU**. (A CPU runtime reproduces the paper's numbers exactly; the code does not need a GPU.)

All files are stored in `My Drive/snn-spike-encoding-rul`, so nothing is lost if Colab disconnects.

## Cell A – one-time setup (run once only)
Mounts Google Drive, unzips the code, installs the exact package versions used in the paper, installs the figure font, and then **restarts the runtime automatically**. After the restart ("Your session crashed" / "restarted" message is expected), continue with **Cell B**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
ROOT = '/content/drive/MyDrive'
FOLDER = 'snn-spike-encoding-rul'
REPO_URL = 'https://github.com/YOUR-USERNAME/snn-spike-encoding-rul.git'   # <- put your GitHub link here
if os.path.exists(f'{ROOT}/{FOLDER}'):
    print('Folder already exists - using it')
elif os.path.exists(f'{ROOT}/AROGYA_N1_code.zip'):
    zipfile.ZipFile(f'{ROOT}/AROGYA_N1_code.zip').extractall(ROOT); os.rename(f'{ROOT}/AROGYA_N1', f'{ROOT}/{FOLDER}')
    print('Unzipped the code from My Drive')
else:
    os.system(f'git clone {REPO_URL} {ROOT}/{FOLDER}'); print('Cloned from GitHub')
!pip -q install jax==0.11.2 jaxlib==0.11.2 optax==0.2.8 numpy==2.4.4 scipy==1.17.1 matplotlib==3.10.8
!apt-get -qq install -y fonts-liberation > /dev/null
!rm -rf ~/.cache/matplotlib
print('Installed. Restarting the runtime now - then run Cell B.')
os.kill(os.getpid(), 9)


## Cell B – start of every session (run after Cell A, and again after any disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/snn-spike-encoding-rul
import jax, numpy, optax, scipy, matplotlib
print('jax', jax.__version__, '| numpy', numpy.__version__, '| optax', optax.__version__, '| scipy', scipy.__version__, '| matplotlib', matplotlib.__version__)
print('device:', jax.devices())
from IPython.display import Image, display
import pandas as pd

Expected: `jax 0.11.2 | numpy 2.4.4 | optax 0.2.8 | scipy 1.17.1 | matplotlib 3.10.8` and a `CpuDevice`.

## Step 2 – download the NASA C-MAPSS data (1 minute)

In [ ]:
!python step2_download_data.py

## Step 3 – installation check (1 minute)
The last line must say *OK – installation works*.

In [ ]:
!python step3_quick_test.py

## Step 4 – Figure 1: the seven encodings
➜ **Insert in the manuscript: Section 3.3, Fig. 1**, after the paragraph that introduces the seven encodings.

In [ ]:
!python step4_fig1_encodings.py
display(Image('figures/Fig1_encodings.png', width=800))

## Step 5 – Figure 2: network architecture
➜ **Insert: Section 3.4, Fig. 2**, after the network equations.

In [ ]:
!python step5_fig2_architecture.py
display(Image('figures/Fig2_architecture.png', width=900))

## Step 6 – Table 3 and Figure 4: information loss of each encoding (1 minute)
➜ **Insert: Table 3 in Section 3.3; Fig. 4 in Section 5.2.**
Expected FD001 values: rate 698 spikes / 0.232, TTFS 420 / 0.072, population 2520 / 0.048, send-on-delta 343 / 0.028, SoD-anchored 357 / 0.077, level-crossing 230 / 0.036.

In [ ]:
!python step6_table3_fig4_information.py
display(pd.read_csv('tables/Table3_encodings.csv'))
display(Image('figures/Fig4_information.png', width=500))

## Step 7 – run all 54 experiments (about 2–3 hours on Colab CPU)
- One line is printed per finished model, e.g. `FD001 lstm 0 rmse 14.67 nasa 293 ep 29 16s`.
- Progress is saved to Google Drive after **every** model. If Colab disconnects: run **Cell B** again, then this cell again – it continues where it stopped.
- Keep this browser tab open. Free Colab may disconnect after long idle periods; simply resume.

In [ ]:
!python step7_run_experiments.py

## Step 8 – summary and check
Compare the printout with `expected_output/step8_analyze_output.txt` (shown below the new results).

In [ ]:
!python step8_analyze.py
print('\n===== EXPECTED (from the paper) =====')
print(open('expected_output/step8_analyze_output.txt').read())

## Step 9 – Tables 5 to 8
➜ **Insert:** Table 5 in Section 5.1 · Table 6 in Section 5.3 · Table 7 in Section 5.5 · Table 8 in Section 5.6.

In [ ]:
!python step9_tables_5_to_8.py > /dev/null
for t in ['Table5_accuracy', 'Table6_ranks', 'Table7_efficiency', 'Table8_robustness']:
    print('\n' + t); display(pd.read_csv(f'tables/{t}.csv'))

## Step 10 – Figures 3, 5 and 6
➜ **Insert:** Fig. 3 in Section 5.1 (after Table 5) · Fig. 5 in Section 5.5 (after Table 7) · Fig. 6 in Section 5.6 (after Table 8).

In [ ]:
!python step10_figs_3_5_6.py
for f in ['Fig3_predictions', 'Fig5_pareto', 'Fig6_robustness']:
    print(f); display(Image(f'figures/{f}.png', width=850))

## Step 11 – download all figures and tables to your computer
They are also already in `My Drive/snn-spike-encoding-rul/figures` and `/tables`.

In [ ]:
!zip -qr AROGYA_N1_results.zip figures tables summary.pkl results.pkl
from google.colab import files
files.download('AROGYA_N1_results.zip')